## Finetune your own Speech-to-Text Whisper model on the language of your choice on a GPU, for free!

### Setup GPU
First, you'll need to enable GPUs for the notebook: Navigate to Edit→Notebook Settings Select T4 GPU from the Hardware Accelerator section Click Save and accept. Next, we'll confirm that we can connect to the GPU:

In [7]:
import torch

if not torch.cuda.is_available():
    print("GPU NOT available!")
else:
    print("GPU is available!")

GPU is available!


### Setup and login Hugging Face

The dataset we use for finetuning is Mozilla's [Common Voice](https://commonvoice.mozilla.org/).

In order to download the Common Voice dataset, track training and evaluation metrics of the finetuning and save your final model to use it and share it with others later, we will be using the Hugging Face (HF) platform. Before starting, make sure you:
1. have a HF [account](https://huggingface.co/join)
2. set up [personal access token](huggingface.co/settings/tokens)
3. login to hugging face in this notebook by running the command below and using your token


In [8]:
!hf auth login


User is already logged in. Use `hf auth login --force` to force re-login.


### Download and install speech-to-text-finetune package

In [9]:
!git clone https://github.com/mozilla-ai/speech-to-text-finetune.git

Cloning into 'speech-to-text-finetune'...
remote: Enumerating objects: 1177, done.
remote: Counting objects: 100% (443/443), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 1177 (delta 349), reused 231 (delta 231), pack-reused 734 (from 2)
Receiving objects: 100% (1177/1177), 5.83 MiB | 8.39 MiB/s, done.
Resolving deltas: 100% (573/573), done.


In [10]:
%cd speech-to-text-finetune/

/content/speech-to-text-finetune/speech-to-text-finetune


In [11]:
!pip install --quiet -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for speech-to-text-finetune (pyproject.toml) ... done


***IMPORTANT:*** After installing the package, you need to restart the kernel / session: "Runtime -> Restart session" and then run the cells below

In [12]:
%cd speech-to-text-finetune/  # after restarting the session, you will need to change directory again

[Errno 2] No such file or directory: 'speech-to-text-finetune/ # after restarting the session, you will need to change directory again'
/content/speech-to-text-finetune/speech-to-text-finetune


In [13]:
from speech_to_text_finetune.finetune_whisper import run_finetuning

In [24]:
!mkdir -p /content/commonvoice/
!tar -xzf "/content/1781705178557-cv-corpus-26.0-2026-06-12-te.tar.gz" -C /content/commonvoice/

In [25]:
!find /content/commonvoice -maxdepth 3

/content/commonvoice
/content/commonvoice/cv-corpus-26.0-2026-06-12
/content/commonvoice/cv-corpus-26.0-2026-06-12/te
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/README.md
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/unvalidated_sentences.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/validated.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/train.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/other.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/reported.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/clips
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/test.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/clip_durations.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/invalidated.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/dev.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/validated_sentences.tsv


In [26]:
%%writefile /content/config.yaml
dataset_id: /content/commonvoice/cv-corpus-26.0-2026-06-12/te
language: Telugu
model_id: openai/whisper-small
n_test_samples: -1
n_train_samples: -1
repo_name: default
training_hp:
  eval_strategy: steps
  fp16: true
  generation_max_length: 225
  gradient_accumulation_steps: 1
  gradient_checkpointing: true
  greater_is_better: false
  hub_private_repo: true
  learning_rate: 1.0e-05
  load_best_model_at_end: true
  logging_steps: 5
  max_steps: 50
  metric_for_best_model: wer
  per_device_eval_batch_size: 8
  per_device_train_batch_size: 32
  predict_with_generate: true
  push_to_hub: false
  save_steps: 5
  save_total_limit: 1
  warmup_steps: 50

Overwriting /content/config.yaml


**NOTE**: Certain "high-resource" languages like English or French have really big datasets (+50GB) which might fill up your disk storage fast. Make sure you have enough storage available before choosing a Common Voice language and finetuning on it.

In [27]:
# @title Finetuning configuration and hyperparameter setting
import yaml


def save_to_yaml(filename="config.yaml"):
    with open(filename, "w") as file:
        yaml.dump(cfg, file)


model_id = "openai/whisper-small"  # @param ["openai/whisper-tiny", "openai/whisper-small", "openai/whisper-medium","openai/whisper-large-v3"]
dataset_id = "mozilla-foundation/common_voice_17_0"  # @param {type: "string"}
language = "Hindi"  # @param {type: "string"}
repo_name = "default"  # @param {type: "string"}
push_to_hub = True  # @param {type: 'boolean'}
n_train_samples = -1  # @param {type: "int"}
n_test_samples = -1  # @param {type: "int"}
hub_private_repo = True  # @param {type: 'boolean'}
max_steps = 50  # @param {type: "slider", min: 1, max: 3000, step: 10}
per_device_train_batch_size = 32  # @param {type: "slider", min: 1, max: 300}
gradient_accumulation_steps = 1  # @param {type: "slider", min: 1, max: 10}
warmup_steps = 50  # @param {type: "slider", min: 0, max: 500}
gradient_checkpointing = True  # @param {type: 'boolean'}
fp16 = True  # @param {type: 'boolean'}
per_device_eval_batch_size = 8  # @param {type: "slider", min: 1, max: 200}
save_steps = 5  # @param {type: "slider", min: 1, max: 500}
logging_steps = 5  # @param {type: "slider", min: 1, max: 500}
load_best_model_at_end = True  # @param {type: 'boolean'}

cfg = {
    "model_id": model_id,
    "dataset_id": dataset_id,
    "language": language,
    "repo_name": repo_name,
    "n_train_samples": n_train_samples,
    "n_test_samples": n_test_samples,
    "training_hp": {
        "push_to_hub": push_to_hub,
        "hub_private_repo": hub_private_repo,
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": 1e-5,
        "warmup_steps": warmup_steps,
        "gradient_checkpointing": gradient_checkpointing,
        "fp16": fp16,
        "eval_strategy": "steps",
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "save_steps": save_steps,
        "logging_steps": logging_steps,
        "load_best_model_at_end": load_best_model_at_end,
        "save_total_limit": 1,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
    },
}

save_to_yaml()

### Start finetuning job

Note that this might take a while, anything from 10min to 10hours depending on your model choice and hyper-parameter configuration

In [28]:
!cat config.yaml

dataset_id: mozilla-foundation/common_voice_17_0
language: Hindi
model_id: openai/whisper-small
n_test_samples: -1
n_train_samples: -1
repo_name: default
training_hp:
  eval_strategy: steps
  fp16: true
  generation_max_length: 225
  gradient_accumulation_steps: 1
  gradient_checkpointing: true
  greater_is_better: false
  hub_private_repo: true
  learning_rate: 1.0e-05
  load_best_model_at_end: true
  logging_steps: 5
  max_steps: 50
  metric_for_best_model: wer
  per_device_eval_batch_size: 8
  per_device_train_batch_size: 32
  predict_with_generate: true
  push_to_hub: true
  save_steps: 5
  save_total_limit: 1
  warmup_steps: 50


In [29]:
from speech_to_text_finetune.config import load_config

cfg = load_config("config.yaml")
print(cfg)

model_id='openai/whisper-small' dataset_id='mozilla-foundation/common_voice_17_0' language='Hindi' repo_name='default' n_train_samples=-1 n_test_samples=-1 training_hp=TrainingConfig(push_to_hub=True, hub_private_repo=True, max_steps=50, per_device_train_batch_size=32, gradient_accumulation_steps=1, learning_rate=1e-05, warmup_steps=50, gradient_checkpointing=True, fp16=True, eval_strategy='steps', per_device_eval_batch_size=8, predict_with_generate=True, generation_max_length=225, save_steps=5, logging_steps=5, load_best_model_at_end=True, save_total_limit=1, metric_for_best_model='wer', greater_is_better=False)


In [30]:
from pathlib import Path

config_path = Path("config.yaml")

print(config_path.read_text())

dataset_id: mozilla-foundation/common_voice_17_0
language: Hindi
model_id: openai/whisper-small
n_test_samples: -1
n_train_samples: -1
repo_name: default
training_hp:
  eval_strategy: steps
  fp16: true
  generation_max_length: 225
  gradient_accumulation_steps: 1
  gradient_checkpointing: true
  greater_is_better: false
  hub_private_repo: true
  learning_rate: 1.0e-05
  load_best_model_at_end: true
  logging_steps: 5
  max_steps: 50
  metric_for_best_model: wer
  per_device_eval_batch_size: 8
  per_device_train_batch_size: 32
  predict_with_generate: true
  push_to_hub: true
  save_steps: 5
  save_total_limit: 1
  warmup_steps: 50



In [31]:
from pathlib import Path

config = """dataset_id: /content/commonvoice/cv-corpus-26.0-2026-06-12/te
language: Telugu
model_id: openai/whisper-small
n_test_samples: -1
n_train_samples: -1
repo_name: default
training_hp:
  eval_strategy: steps
  fp16: true
  generation_max_length: 225
  gradient_accumulation_steps: 1
  gradient_checkpointing: true
  greater_is_better: false
  hub_private_repo: true
  learning_rate: 1.0e-05
  load_best_model_at_end: true
  logging_steps: 5
  max_steps: 50
  metric_for_best_model: wer
  per_device_eval_batch_size: 8
  per_device_train_batch_size: 32
  predict_with_generate: true
  push_to_hub: true
  save_steps: 5
  save_total_limit: 1
  warmup_steps: 50
"""

with open("config.yaml", "w") as f:
    f.write(config)

print("✅ config.yaml updated successfully!")

✅ config.yaml updated successfully!


In [32]:
!cat config.yaml

dataset_id: /content/commonvoice/cv-corpus-26.0-2026-06-12/te
language: Telugu
model_id: openai/whisper-small
n_test_samples: -1
n_train_samples: -1
repo_name: default
training_hp:
  eval_strategy: steps
  fp16: true
  generation_max_length: 225
  gradient_accumulation_steps: 1
  gradient_checkpointing: true
  greater_is_better: false
  hub_private_repo: true
  learning_rate: 1.0e-05
  load_best_model_at_end: true
  logging_steps: 5
  max_steps: 50
  metric_for_best_model: wer
  per_device_eval_batch_size: 8
  per_device_train_batch_size: 32
  predict_with_generate: true
  push_to_hub: true
  save_steps: 5
  save_total_limit: 1
  warmup_steps: 50


In [33]:
run_finetuning(config_path="config.yaml")

2026-07-08 14:58:41.301 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:59 - Finetuning starts soon, results saved locally at ./artifacts/whisper-small-te
2026-07-08 14:58:41.438 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:64 - Results will also be uploaded in HF at meghanavanamoju/whisper-small-te. Private repo is set to True.
2026-07-08 14:58:41.439 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:70 - Loading openai/whisper-small on Tesla T4 and configuring it for Telugu.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

2026-07-08 14:58:43.789 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:104 - Loading /content/commonvoice/cv-corpus-26.0-2026-06-12/te. Language selected Telugu
2026-07-08 14:58:43.826 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:111 - Processing dataset...


Map:   0%|          | 0/99 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Step,Wer Ortho,Wer,Cer Ortho,Cer
0.882964,1.077561,50,103.438395,72.237960,60.631970,43.185690


2026-07-08 15:26:03.146 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:164 - Evaluation complete. Results:
	 {'eval_loss': 1.077560544013977, 'eval_wer_ortho': 103.43839541547277, 'eval_wer': 72.23796033994334, 'eval_cer_ortho': 60.63197026022304, 'eval_cer': 43.18568994889267}
2026-07-08 15:26:03.152 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:178 - Uploading model and eval results to HuggingFace: meghanavanamoju/whisper-small-te


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-08 15:26:27.837 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:185 - Find your final, best performing model at ./artifacts/whisper-small-te


({'eval_loss': 2.7994391918182373,
  'eval_wer_ortho': 197.7077363896848,
  'eval_wer': 211.7091595845137,
  'eval_cer_ortho': 229.88847583643124,
  'eval_cer': 174.02044293015334},
 {'eval_loss': 1.077560544013977,
  'eval_wer_ortho': 103.43839541547277,
  'eval_wer': 72.23796033994334,
  'eval_cer_ortho': 60.63197026022304,
  'eval_cer': 43.18568994889267})

In [82]:
!ls /content/common_voice

ls: cannot access '/content/common_voice': No such file or directory


In [83]:
!ls -lh /content

total 63M
-rw-r--r--  1 root root  63M Jul  8 13:04 1781705178557-cv-corpus-26.0-2026-06-12-te.tar.gz
drwxr-xr-x  3 root root 4.0K Jul  8 13:09 commonvoice
-rw-r--r--  1 root root  656 Jul  8 13:22 config.yaml
drwxr-xr-x  1 root root 4.0K Jun  4 13:32 sample_data
drwxr-xr-x 11 root root 4.0K Jul  8 12:45 speech-to-text-finetune


In [84]:
!find /content/commonvoice -maxdepth 3

/content/commonvoice
/content/commonvoice/cv-corpus-26.0-2026-06-12
/content/commonvoice/cv-corpus-26.0-2026-06-12/te
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/other.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/validated_sentences.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/test.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/reported.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/invalidated.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/unvalidated_sentences.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/dev.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/clips
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/validated.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/README.md
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/clip_durations.tsv
/content/commonvoice/cv-corpus-26.0-2026-06-12/te/train.tsv


In [34]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

CUDA available: True
GPU: Tesla T4
